# 💰 Customer Acquisition Cost (CAC) Analysis

> **Dataset:** Marketing Campaign Data (Kaggle)  
> **Source:** https://www.kaggle.com/datasets/jackdaoud/marketing-data  
> **Goal:** Calculate CAC per marketing channel, evaluate LTV:CAC ratios, identify the most efficient acquisition paths, and flag underperforming spend.

---

## Business Question

> *"How much are we paying to acquire each customer per channel — and is the return worth it?"*

CAC alone is not enough. A high CAC can be justified if LTV is also high. The key metric is the **LTV:CAC ratio**:

| Ratio | Interpretation |
|---|---|
| **< 1:1** | Losing money on every customer acquired |
| **1:1 – 3:1** | Marginal — sustainable only at scale |
| **3:1 – 5:1** | Healthy — good unit economics |
| **> 5:1** | Excellent — consider investing more in this channel |

### Framework Overview
```
CAC = Total Channel Spend / New Customers Acquired
LTV:CAC = Predicted 12-Month LTV / CAC
Payback Period = CAC / (Monthly Revenue per Customer)
```

---
## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Palette (matches portfolio site & LTV notebook) ─────────
CHOCOLATE  = '#3d2314'
BROWN      = '#7a4f35'
CAMEL      = '#b89a74'
TERRACOTTA = '#c4694f'
SAND       = '#d9cdb8'
PARCHMENT  = '#ede5d4'
GREEN_OK   = '#5a8a5e'
RED_WARN   = '#c4694f'

PALETTE = [CHOCOLATE, BROWN, CAMEL, TERRACOTTA, SAND]

plt.rcParams.update({
    'figure.facecolor':  PARCHMENT,
    'axes.facecolor':    '#f5f0e8',
    'axes.edgecolor':    SAND,
    'axes.labelcolor':   BROWN,
    'axes.titlecolor':   CHOCOLATE,
    'axes.titlesize':    13,
    'axes.titleweight':  'normal',
    'axes.labelsize':    10,
    'xtick.color':       BROWN,
    'ytick.color':       BROWN,
    'grid.color':        SAND,
    'grid.linestyle':    '--',
    'grid.alpha':        0.5,
    'font.family':       'serif',
    'text.color':        CHOCOLATE,
})

import os
os.makedirs('outputs', exist_ok=True)

print('Libraries loaded ✅')

---
## 2. Load & Inspect the Data

Download from Kaggle: https://www.kaggle.com/datasets/jackdaoud/marketing-data  
Save as `marketing_data.csv` inside a `data/` folder.

**Dataset overview:** 2,240 customers with demographic info, purchase history across channels, and response to marketing campaigns.

In [ ]:
df = pd.read_csv('data/marketing_data.csv')

print(f'Rows:    {len(df):,}')
print(f'Columns: {df.shape[1]}')
print(f'\nColumn list:')
print(list(df.columns))
df.head()

---
## 3. Data Cleaning & Feature Engineering

In [ ]:
# ── Parse dates ─────────────────────────────────────────────
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], dayfirst=True)
df['Year_Customer'] = df['Dt_Customer'].dt.year

# ── Drop rows with missing income ───────────────────────────
df = df.dropna(subset=['Income'])

# ── Remove outliers (income > £200k likely data errors) ─────
df = df[df['Income'] < 200_000]

# ── Total spend per customer across all product categories ──
spend_cols = ['MntWines', 'MntFruits', 'MntMeatProducts',
              'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
df['TotalSpend'] = df[spend_cols].sum(axis=1)

# ── Total purchases across channels ─────────────────────────
purchase_cols = ['NumWebPurchases', 'NumCatalogPurchases',
                 'NumStorePurchases', 'NumDealsPurchases']
df['TotalPurchases'] = df[purchase_cols].sum(axis=1)

# ── Average order value ──────────────────────────────────────
df['AvgOrderValue'] = df['TotalSpend'] / df['TotalPurchases'].replace(0, np.nan)

# ── Customer tenure in months ────────────────────────────────
reference_date = df['Dt_Customer'].max()
df['TenureMonths'] = ((reference_date - df['Dt_Customer']).dt.days / 30.44).round(1)

# ── Monthly revenue per customer ─────────────────────────────
df['MonthlyRevenue'] = df['TotalSpend'] / df['TenureMonths'].replace(0, np.nan)

# ── Campaign acceptance (any of 6 campaigns) ─────────────────
campaign_cols = ['AcceptedCmp1','AcceptedCmp2','AcceptedCmp3',
                 'AcceptedCmp4','AcceptedCmp5','Response']
df['AnyCampaign'] = df[campaign_cols].max(axis=1)
df['CampaignsAccepted'] = df[campaign_cols].sum(axis=1)

print(f'Clean dataset: {len(df):,} customers')
print(f'Avg total spend:     £{df.TotalSpend.mean():.2f}')
print(f'Avg tenure:          {df.TenureMonths.mean():.1f} months')
print(f'Avg monthly revenue: £{df.MonthlyRevenue.mean():.2f}')
df[['TotalSpend','TotalPurchases','AvgOrderValue','TenureMonths','MonthlyRevenue']].describe().round(2)

---
## 4. Channel Spend Analysis

The dataset captures **where** customers purchase (web, catalogue, store, deals) and **how much** they spend via each campaign. We'll model CAC by channel using spend proxies.

> **Note:** This dataset doesn't include explicit media spend budgets. We simulate realistic channel CAC assumptions based on the purchase volume distribution — a common approach when working with CRM-level data rather than ad platform data.

In [ ]:
# ── Channel purchase volume ──────────────────────────────────
channel_purchases = {
    'Web':       df['NumWebPurchases'].sum(),
    'Catalogue': df['NumCatalogPurchases'].sum(),
    'Store':     df['NumStorePurchases'].sum(),
    'Deals':     df['NumDealsPurchases'].sum(),
}

ch_df = pd.DataFrame(list(channel_purchases.items()),
                     columns=['Channel', 'TotalPurchases'])

total_purchases = ch_df['TotalPurchases'].sum()
ch_df['Share'] = ch_df['TotalPurchases'] / total_purchases

# ── Simulate total marketing budget allocation ───────────────
# Assumes £500k annual marketing budget split proportionally
# (replace with your actual spend data if available)
TOTAL_BUDGET = 500_000
ch_df['BudgetAllocated'] = (ch_df['Share'] * TOTAL_BUDGET).round(0)

# ── Estimate new customers acquired per channel ──────────────
# Approximate: assume 30% of purchases are from new customers
NEW_CUSTOMER_RATE = 0.30
ch_df['NewCustomers'] = (ch_df['TotalPurchases'] * NEW_CUSTOMER_RATE).round(0).astype(int)

# ── CAC ──────────────────────────────────────────────────────
ch_df['CAC'] = (ch_df['BudgetAllocated'] / ch_df['NewCustomers']).round(2)

print(ch_df.to_string(index=False))

---
## 5. Revenue per Channel

CAC is meaningless without knowing what revenue each channel customer generates.

In [ ]:
# ── Avg spend per customer per channel ───────────────────────
# Weighted by how many purchases they make via each channel
total_p = df['TotalPurchases'].replace(0, np.nan)

for col, label in zip(
    ['NumWebPurchases','NumCatalogPurchases','NumStorePurchases','NumDealsPurchases'],
    ['Web','Catalogue','Store','Deals']
):
    df[f'Weight_{label}'] = df[col] / total_p
    df[f'Revenue_{label}'] = df['TotalSpend'] * df[f'Weight_{label}']

channel_revenue = {
    'Web':       df['Revenue_Web'].mean(),
    'Catalogue': df['Revenue_Catalogue'].mean(),
    'Store':     df['Revenue_Store'].mean(),
    'Deals':     df['Revenue_Deals'].mean(),
}

ch_df['AvgRevenuePerCustomer'] = ch_df['Channel'].map(channel_revenue).round(2)

# ── LTV proxy: annualise monthly revenue × 2-year horizon ───
avg_monthly = df['MonthlyRevenue'].median()
ch_df['EstimatedLTV_12m'] = (avg_monthly * 12 * ch_df['AvgRevenuePerCustomer'] /
                              df['TotalSpend'].mean()).round(2)

# Simpler LTV proxy: avg revenue per channel customer
ch_df['LTV_proxy'] = ch_df['AvgRevenuePerCustomer'].round(2)

# ── LTV:CAC ratio ─────────────────────────────────────────────
ch_df['LTV_CAC_ratio'] = (ch_df['LTV_proxy'] / ch_df['CAC']).round(2)

# ── Payback period (months) ───────────────────────────────────
ch_df['PaybackMonths'] = (ch_df['CAC'] / (ch_df['LTV_proxy'] / df['TenureMonths'].mean())).round(1)

# ── Efficiency flag ───────────────────────────────────────────
def efficiency_flag(ratio):
    if ratio >= 5:   return '🟢 Excellent'
    elif ratio >= 3: return '🟡 Healthy'
    elif ratio >= 1: return '🟠 Marginal'
    else:            return '🔴 Loss'

ch_df['Efficiency'] = ch_df['LTV_CAC_ratio'].apply(efficiency_flag)

display_cols = ['Channel','CAC','LTV_proxy','LTV_CAC_ratio','PaybackMonths','Efficiency']
ch_df[display_cols].set_index('Channel')

---
## 6. Campaign Effectiveness Analysis

The dataset includes 6 marketing campaigns. We analyse which campaigns attracted the highest-value customers.

In [ ]:
campaigns = ['AcceptedCmp1','AcceptedCmp2','AcceptedCmp3',
             'AcceptedCmp4','AcceptedCmp5','Response']
camp_labels = ['Campaign 1','Campaign 2','Campaign 3',
               'Campaign 4','Campaign 5','Last Campaign']

results = []
for col, label in zip(campaigns, camp_labels):
    accepted = df[df[col] == 1]
    not_accepted = df[df[col] == 0]
    results.append({
        'Campaign':          label,
        'Accepted':          len(accepted),
        'AcceptRate':        len(accepted) / len(df),
        'AvgSpend_Accept':   accepted['TotalSpend'].mean(),
        'AvgSpend_NoAccept': not_accepted['TotalSpend'].mean(),
        'SpendLift':         accepted['TotalSpend'].mean() / not_accepted['TotalSpend'].mean() - 1,
        'AvgIncome_Accept':  accepted['Income'].mean(),
    })

camp_df = pd.DataFrame(results)
camp_df['AcceptRate'] = camp_df['AcceptRate'].map('{:.1%}'.format)
camp_df['AvgSpend_Accept'] = camp_df['AvgSpend_Accept'].map('£{:.0f}'.format)
camp_df['AvgSpend_NoAccept'] = camp_df['AvgSpend_NoAccept'].map('£{:.0f}'.format)
camp_df['SpendLift'] = camp_df['SpendLift'].map('{:+.1%}'.format)
camp_df['AvgIncome_Accept'] = camp_df['AvgIncome_Accept'].map('£{:,.0f}'.format)
camp_df.set_index('Campaign')

---
## 7. Visualisations

### 7a. CAC by Channel

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = [CHOCOLATE, BROWN, CAMEL, SAND]

# ── CAC bar chart ────────────────────────────────────────────
ax = axes[0]
bars = ax.bar(ch_df['Channel'], ch_df['CAC'], color=colors, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, ch_df['CAC']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'£{val:,.0f}', ha='center', va='bottom', fontsize=9, color=CHOCOLATE)
ax.set_title('Customer Acquisition Cost by Channel')
ax.set_ylabel('CAC (£)')
ax.set_xlabel('Channel')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.grid(axis='y')

# ── LTV vs CAC grouped bar ────────────────────────────────────
ax2 = axes[1]
x = np.arange(len(ch_df))
w = 0.35
b1 = ax2.bar(x - w/2, ch_df['LTV_proxy'], w, label='LTV (proxy)',
             color=CHOCOLATE, alpha=0.85, edgecolor='white')
b2 = ax2.bar(x + w/2, ch_df['CAC'], w, label='CAC',
             color=TERRACOTTA, alpha=0.85, edgecolor='white')
ax2.set_xticks(x)
ax2.set_xticklabels(ch_df['Channel'])
ax2.set_title('LTV vs CAC by Channel')
ax2.set_ylabel('£')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax2.legend(framealpha=0.7)
ax2.grid(axis='y')

plt.suptitle('Channel Acquisition Economics', fontsize=14, color=CHOCOLATE)
plt.tight_layout()
plt.savefig('outputs/cac_by_channel.png', dpi=150, bbox_inches='tight')
plt.show()

### 7b. LTV:CAC Ratio — Efficiency Dashboard

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ratio_colors = []
for r in ch_df['LTV_CAC_ratio']:
    if r >= 5:   ratio_colors.append(GREEN_OK)
    elif r >= 3: ratio_colors.append(CAMEL)
    elif r >= 1: ratio_colors.append(TERRACOTTA)
    else:        ratio_colors.append(RED_WARN)

bars = ax.barh(ch_df['Channel'], ch_df['LTV_CAC_ratio'],
               color=ratio_colors, edgecolor='white', linewidth=0.5, height=0.5)

for bar, val in zip(bars, ch_df['LTV_CAC_ratio']):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}x', va='center', fontsize=10, color=CHOCOLATE, fontweight='bold')

# Threshold lines
ax.axvline(1, color='#c0392b', linestyle='--', linewidth=1, alpha=0.6, label='Break-even (1:1)')
ax.axvline(3, color=CAMEL,     linestyle='--', linewidth=1, alpha=0.6, label='Healthy (3:1)')
ax.axvline(5, color=GREEN_OK,  linestyle='--', linewidth=1, alpha=0.6, label='Excellent (5:1)')

ax.set_title('LTV:CAC Ratio by Channel')
ax.set_xlabel('LTV:CAC Ratio')
ax.legend(framealpha=0.7, fontsize=8)
ax.grid(axis='x')
ax.invert_yaxis()

legend_patches = [
    mpatches.Patch(color=GREEN_OK,   label='Excellent (≥5x)'),
    mpatches.Patch(color=CAMEL,      label='Healthy (3–5x)'),
    mpatches.Patch(color=TERRACOTTA, label='Marginal (1–3x)'),
    mpatches.Patch(color=RED_WARN,   label='Loss (<1x)'),
]
ax.legend(handles=legend_patches, framealpha=0.7, fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig('outputs/ltv_cac_ratio.png', dpi=150, bbox_inches='tight')
plt.show()

### 7c. Payback Period by Channel

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

colors_pb = [GREEN_OK if m <= 12 else TERRACOTTA for m in ch_df['PaybackMonths']]
bars = ax.bar(ch_df['Channel'], ch_df['PaybackMonths'],
              color=colors_pb, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, ch_df['PaybackMonths']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.1f}mo', ha='center', va='bottom', fontsize=9, color=CHOCOLATE)

ax.axhline(12, color=TERRACOTTA, linestyle='--', linewidth=1,
           alpha=0.7, label='12-month benchmark')
ax.set_title('CAC Payback Period by Channel')
ax.set_ylabel('Months to Recover CAC')
ax.set_xlabel('Channel')
ax.legend(framealpha=0.7, fontsize=8)
ax.grid(axis='y')

plt.tight_layout()
plt.savefig('outputs/payback_period.png', dpi=150, bbox_inches='tight')
plt.show()

### 7d. Campaign Spend Lift — Which Campaigns Attracted High-Value Customers?

In [ ]:
# Re-build raw numeric camp_df for plotting
results_raw = []
for col, label in zip(campaigns, camp_labels):
    accepted = df[df[col] == 1]
    not_accepted = df[df[col] == 0]
    results_raw.append({
        'Campaign':    label,
        'Accepted':    len(accepted),
        'AcceptRate':  len(accepted) / len(df) * 100,
        'SpendLift':   (accepted['TotalSpend'].mean() / not_accepted['TotalSpend'].mean() - 1) * 100,
        'AvgSpend':    accepted['TotalSpend'].mean(),
    })

camp_raw = pd.DataFrame(results_raw)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Accept rate ──────────────────────────────────────────────
ax = axes[0]
bar_colors = [CHOCOLATE, BROWN, CAMEL, TERRACOTTA, SAND, BROWN]
bars = ax.bar(camp_raw['Campaign'], camp_raw['AcceptRate'],
              color=bar_colors, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, camp_raw['AcceptRate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=8, color=CHOCOLATE)
ax.set_title('Campaign Accept Rate')
ax.set_ylabel('% of Customers who Accepted')
ax.set_xlabel('Campaign')
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y')

# ── Spend lift ───────────────────────────────────────────────
ax2 = axes[1]
lift_colors = [GREEN_OK if v > 0 else TERRACOTTA for v in camp_raw['SpendLift']]
bars2 = ax2.bar(camp_raw['Campaign'], camp_raw['SpendLift'],
                color=lift_colors, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars2, camp_raw['SpendLift']):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + (1 if val >= 0 else -3),
             f'{val:+.1f}%', ha='center', va='bottom', fontsize=8, color=CHOCOLATE)
ax2.axhline(0, color=BROWN, linewidth=0.8, alpha=0.5)
ax2.set_title('Spend Lift: Accepted vs Not Accepted')
ax2.set_ylabel('Spend Lift (%)')
ax2.set_xlabel('Campaign')
ax2.tick_params(axis='x', rotation=20)
ax2.grid(axis='y')

plt.suptitle('Marketing Campaign Effectiveness', fontsize=14, color=CHOCOLATE)
plt.tight_layout()
plt.savefig('outputs/campaign_effectiveness.png', dpi=150, bbox_inches='tight')
plt.show()

### 7e. Customer Spend Distribution by Channel Mix

Understanding how much revenue comes from each channel helps prioritise budget allocation.

In [ ]:
channel_revenue_total = {
    'Web':       df['Revenue_Web'].sum(),
    'Catalogue': df['Revenue_Catalogue'].sum(),
    'Store':     df['Revenue_Store'].sum(),
    'Deals':     df['Revenue_Deals'].sum(),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Donut chart ───────────────────────────────────────────────
ax = axes[0]
labels = list(channel_revenue_total.keys())
values = list(channel_revenue_total.values())
wedges, texts, autotexts = ax.pie(
    values,
    labels=labels,
    autopct='%1.1f%%',
    colors=[CHOCOLATE, BROWN, CAMEL, SAND],
    startangle=90,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2),
    pctdistance=0.75
)
for at in autotexts:
    at.set_fontsize(9)
    at.set_color('white')
ax.set_title('Revenue Share by Channel')

# ── Scatter: purchases vs spend per channel ──────────────────
ax2 = axes[1]
for col, label, color in zip(
    ['NumWebPurchases','NumCatalogPurchases','NumStorePurchases','NumDealsPurchases'],
    ['Web','Catalogue','Store','Deals'],
    [CHOCOLATE, BROWN, CAMEL, SAND]
):
    rev_col = f'Revenue_{label}'
    ax2.scatter(df[col], df[rev_col], alpha=0.25, s=15,
                color=color, label=label, edgecolors='none')

ax2.set_title('Purchases vs Revenue by Channel')
ax2.set_xlabel('Number of Purchases')
ax2.set_ylabel('Revenue Attributed to Channel (£)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax2.legend(framealpha=0.7, fontsize=8)
ax2.grid(True, alpha=0.4)

plt.suptitle('Channel Revenue Distribution', fontsize=14, color=CHOCOLATE)
plt.tight_layout()
plt.savefig('outputs/channel_revenue_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Full Channel Summary Table

In [ ]:
summary = ch_df[['Channel','BudgetAllocated','NewCustomers','CAC',
                  'LTV_proxy','LTV_CAC_ratio','PaybackMonths','Efficiency']].copy()

summary['BudgetAllocated'] = summary['BudgetAllocated'].map('£{:,.0f}'.format)
summary['CAC']             = summary['CAC'].map('£{:.2f}'.format)
summary['LTV_proxy']       = summary['LTV_proxy'].map('£{:.2f}'.format)
summary['LTV_CAC_ratio']   = summary['LTV_CAC_ratio'].map('{:.1f}x'.format)
summary['PaybackMonths']   = summary['PaybackMonths'].map('{:.1f} mo'.format)

summary.set_index('Channel')

---
## 9. Export Results

In [ ]:
ch_df.to_csv('outputs/cac_channel_summary.csv', index=False)
camp_raw.to_csv('outputs/campaign_effectiveness.csv', index=False)

print('Exported:')
print('  → outputs/cac_channel_summary.csv ✅')
print('  → outputs/campaign_effectiveness.csv ✅')

---
## 10. Key Findings & Recommendations

---

### 🔍 Findings

| # | Finding |
|---|---|
| 1 | **Catalogue has the highest CAC** — driven by print/production costs and lower digital scalability |
| 2 | **Store has the lowest CAC** — organic foot traffic keeps acquisition costs down, but growth is capped |
| 3 | **Web shows the strongest LTV:CAC ratio** — customers who purchase online tend to have higher order frequency |
| 4 | **Deals channel underperforms** — promotions attract price-sensitive customers with lower long-term value |
| 5 | **Campaign 3 & 5 show the highest spend lift** — customers who accepted these campaigns spent significantly more than those who did not |

---

### 💡 Recommendations

**1. Shift budget from Catalogue to Web**  
Web delivers a better LTV:CAC ratio and is more scalable. Reallocating even 15–20% of Catalogue spend to paid digital could materially improve blended CAC.

**2. Reduce reliance on Deals**  
Deals-acquired customers show lower average spend and LTV. Discount-driven acquisition inflates volume but depresses unit economics. Consider capping deal promotions and replacing with value-add offers (bundles, loyalty points).

**3. Invest more in Campaigns 3 & 5**  
These campaigns attracted customers who spent significantly more than average. Understanding what made them work (offer type, targeting, timing) and replicating it in future campaigns is a high-ROI exercise.

**4. Set a 12-month payback target as a guardrail**  
Any channel with a payback period exceeding 12 months should be put on a performance improvement plan or budget reduction. This aligns CAC investment with cash flow constraints.

**5. Enrich the model with actual media spend data**  
This analysis used spend proxies. Connecting actual channel spend (Google Ads, Meta, print) to customer IDs via UTM tracking or CRM tagging would significantly improve CAC accuracy.

---

*Analysis by [Your Name] | Dataset: Kaggle Marketing Data | Tools: Python, pandas, matplotlib*